# Value at Risk

Follow one risk number from market data to a plain-English answer. You read the TimesFM forecast, trace a Monte Carlo path, compare forward-looking and historical VaR, add a stress scenario, and ask the data agent about it.

This notebook is the same walkthrough as [`docs/01-value-at-risk.md`](https://github.com/gCloud-Tech-Showcase/datacloud-market-risk/blob/main/docs/01-value-at-risk.md), with charts. Run the cells in order. Your numbers differ slightly from the guide, because TimesFM does not always return identical intervals for identical input.

### Before you run it

| | |
| --- | --- |
| **Run time** | About 3 minutes top to bottom. Step 5 writes 3,000,000 rows in 10 to 20 seconds. Everything else reads tables the pipeline already built. |
| **Changes it makes** | Step 5 adds a `tech_down_10` scenario. **Reset**, at the end, removes it. Both are safe to run more than once. |
| **Charts** | Interactive. Hover any bar or dot for exact values. The caption above a chart says what to look for. |

### Five terms you'll see

| Term | Plain meaning |
| --- | --- |
| **VaR (1-day, 99%)** | The loss the portfolio exceeds on only 1 day in 100 |
| **Expected Shortfall (ES)** | The average loss on those worst 1 in 100 days |
| **Volatility** | How much a price typically swings, here per year (annualized) |
| **Filtered Historical Simulation** | Replay real past days, rescaled to today's volatility |
| **Scenario** | A what-if shock applied on top of every simulated day, such as "Tech falls 10%" |

## Set up

In [ ]:
import google.auth
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

_, PROJECT_ID = google.auth.default()
print(f"Project: {PROJECT_ID}")

# Shared chart style: Google Cloud colours, one colour per desk. Hover any chart for exact values.
pio.templates["gc"] = go.layout.Template(layout=dict(
    font=dict(family="Google Sans, Roboto, Arial, sans-serif", size=13, color="#3C4043"),
    title=dict(font=dict(size=17, color="#202124"), x=0, xanchor="left"),
    hoverlabel=dict(bgcolor="white", font_size=13),
    margin=dict(l=10, r=30, t=70, b=40),
    legend=dict(orientation="h", y=1.02, yanchor="bottom", x=1, xanchor="right", title=None),
))
pio.templates.default = "plotly_white+gc"
DESK_COLORS = {"Growth Equities": "#1A73E8", "Value Equities": "#188038", "Defensive Equities": "#9AA0A6"}
BLUE, RED, GREY, LIGHT = "#1A73E8", "#D93025", "#9AA0A6", "#BDC1C6"

## Before you start

Check the stress scenarios. A fresh deployment has two, `baseline` and `energy_down_15`. If `tech_down_10` is listed from an earlier run, run the **Reset** section at the end first.

In [ ]:
%%bigquery
SELECT scenario_id, description, sector, shock_pct
FROM `raw.stress_scenarios`
ORDER BY scenario_id

## Step 1: Look at the portfolio

Pulsar Capital, a fictional firm, holds 30 US large-cap stocks across three desks. The prices behind the positions are real.

In [ ]:
%%bigquery portfolio
SELECT desk, book, COUNT(*) AS positions, SUM(notional_usd) AS notional_usd
FROM `raw.portfolio_positions`
GROUP BY desk, book
ORDER BY desk, book

In [ ]:
p = portfolio.sort_values("notional_usd")
fig = px.bar(p, x="notional_usd", y="book", color="desk", orientation="h", color_discrete_map=DESK_COLORS,
             custom_data=["positions"], title=f"${portfolio['notional_usd'].sum() / 1e6:.0f}M across 3 desks and 5 books")
fig.update_traces(texttemplate="$%{x:.2s} · %{customdata[0]} stocks", textposition="outside", cliponaxis=False,
                  hovertemplate="%{y}<br>$%{x:,.0f} · %{customdata[0]} stocks<extra></extra>")
fig.update_layout(height=330, legend_title_text=None, xaxis=dict(title=None, tickformat="$~s", range=[0, p["notional_usd"].max() * 1.3]),
                  yaxis=dict(title=None, categoryorder="array", categoryarray=list(p["book"])))
fig.show()
portfolio

## Step 2: Read the TimesFM forecast

One `AI.FORECAST` query forecasts all 30 stocks. This is its raw output for JPMorgan. Step 1 is dated the day after the as-of date, even on a weekend: TimesFM infers a daily step, so read step 1 as the next trading day.

In [ ]:
%%bigquery jpm_forecast
SELECT DATE(forecast_timestamp) AS step_date,
       forecast_value,
       prediction_interval_lower_bound AS lower_99,
       prediction_interval_upper_bound AS upper_99,
       ai_forecast_status
FROM `serving.timesfm_forecast`
WHERE symbol = 'JPM'
ORDER BY forecast_timestamp

In [ ]:
jpm_forecast

In [ ]:
%%bigquery jpm_history
SELECT date, close
FROM `processed.daily_returns`
WHERE symbol = 'JPM'
ORDER BY date DESC
LIMIT 60

In [ ]:
history = jpm_history.sort_values("date")
steps = pd.to_datetime(jpm_forecast["step_date"])
fig = go.Figure([
    go.Scatter(mode="lines", x=steps, y=jpm_forecast["upper_99"], line=dict(width=0), showlegend=False,
               hovertemplate="99% high $%{y:.2f}<extra></extra>"),
    go.Scatter(mode="lines", x=steps, y=jpm_forecast["lower_99"], fill="tonexty", fillcolor="rgba(26,115,232,0.15)", line=dict(width=0),
               name="99% band", hovertemplate="99% low $%{y:.2f}<extra></extra>"),
    go.Scatter(mode="lines", x=pd.to_datetime(history["date"]), y=history["close"], name="Close, last 60 days",
               line=dict(color="#5F6368", width=2), hovertemplate="%{x|%b %d}: $%{y:.2f}<extra></extra>"),
    go.Scatter(mode="lines", x=steps, y=jpm_forecast["forecast_value"], name="TimesFM median", line=dict(color=BLUE, width=2.5, dash="dot"),
               hovertemplate="%{x|%b %d} median: $%{y:.2f}<extra></extra>"),
])
fig.update_layout(title="JPM: TimesFM forecast and its 99% band", height=400, hovermode="x unified",
                  yaxis=dict(title=None, tickprefix="$"), xaxis_title=None)
fig.show()

The simulation needs a volatility, not a price. `serving.timesfm_forward_params` turns the 99% interval for step 1 into a 1-day volatility: `ln(upper / lower) / (2 × 2.5758)`.

In [ ]:
%%bigquery vols
SELECT symbol, sector,
       ROUND(100 * timesfm_vol_annualized, 1) AS timesfm_vol_pct,
       ROUND(100 * historical_vol_annualized, 1) AS historical_vol_pct
FROM `serving.timesfm_forward_params`
ORDER BY sector, symbol

**Reading the chart:** each line runs from a stock's trailing one-year volatility (grey dot) to TimesFM's forward view (coloured dot). Blue means TimesFM expects a calmer day than last year, red a jumpier one. Stocks are sorted by the gap.

In [ ]:
v = vols.assign(diff=vols["timesfm_vol_pct"] - vols["historical_vol_pct"]).sort_values("diff")
calmer = int((v["diff"] < 0).sum())


def segments(rows):
    # One line per stock, from last year's volatility to TimesFM's, as a single trace with gaps.
    xs, ys = [], []
    for r in rows.itertuples():
        xs += [r.historical_vol_pct, r.timesfm_vol_pct, None]
        ys += [r.symbol, r.symbol, None]
    return xs, ys


lower, higher = v[v["diff"] < 0], v[v["diff"] >= 0]
fig = go.Figure()
for rows, color in ((lower, BLUE), (higher, RED)):
    xs, ys = segments(rows)
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color=color, width=3), opacity=0.4,
                             showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=v["historical_vol_pct"], y=v["symbol"], mode="markers", name="Trailing 1 year",
                         marker=dict(color=GREY, size=9), hovertemplate="%{y}: %{x:.1f}% last year<extra></extra>"))
for rows, color, label in ((lower, BLUE, "TimesFM, calmer"), (higher, RED, "TimesFM, jumpier")):
    fig.add_trace(go.Scatter(x=rows["timesfm_vol_pct"], y=rows["symbol"], mode="markers", name=label,
                             marker=dict(color=color, size=11), customdata=rows[["historical_vol_pct", "diff", "sector"]],
                             hovertemplate="<b>%{y}</b> · %{customdata[2]}<br>TimesFM %{x:.1f}% vs %{customdata[0]:.1f}% last year"
                                           "<br>%{customdata[1]:+.1f} points<extra></extra>"))
fig.update_layout(title=f"TimesFM expects a calmer day than last year for {calmer} of {len(v)} stocks", height=780,
                  xaxis=dict(title="Annualized volatility", ticksuffix="%"),
                  yaxis=dict(title=None, categoryorder="array", categoryarray=list(v["symbol"])))
fig.show()

For most stocks, TimesFM's band implies lower volatility than the trailing year. The demo uses the band as a volatility input. It does not claim the forward view is more accurate.

## Step 3: Trace one Monte Carlo path

Each of the 100,000 paths replays one of the last 1,000 trading days for all 30 stocks together, rescaled to today's TimesFM volatility. The paths are a smoothed bootstrap of those 1,000 days: the count makes the distribution smooth, not better informed. Path 42 replays one day. Its portfolio P&L is the sum of the 30 rows.

In [ ]:
%%bigquery path_42
SELECT historical_date, symbol, desk, notional_usd,
       ROUND(sim_log_return, 5) AS sim_log_return, ROUND(sim_pnl_usd, 2) AS sim_pnl_usd
FROM `serving.mc_simulated_pnl`
WHERE scenario_id = 'baseline' AND path_id = 42
ORDER BY symbol

In [ ]:
print(f"Path 42 replays {path_42['historical_date'].iloc[0]}. Portfolio P&L: ${path_42['sim_pnl_usd'].sum():,.0f}")
path_42.head(10)

In [ ]:
p42 = path_42.sort_values("sim_pnl_usd")
fig = px.bar(p42, x="sim_pnl_usd", y="symbol", color="desk", orientation="h", color_discrete_map=DESK_COLORS,
             title=f"Path 42 replays {p42['historical_date'].iloc[0]}: 30 stocks, portfolio P&L ${p42['sim_pnl_usd'].sum() / 1e6:+.2f}M")
fig.update_traces(hovertemplate="%{y}: $%{x:,.0f}<extra></extra>")
fig.update_layout(height=660, legend_title_text=None, xaxis=dict(title=None, tickformat="$~s"),
                  yaxis=dict(title=None, categoryorder="array", categoryarray=list(p42["symbol"])))
fig.show()

## Step 4: Compare forward-looking and historical VaR

VaR at 99% is the loss the portfolio exceeds on only 1 path in 100. Expected Shortfall (ES) is the average loss on those worst paths.

In [ ]:
%%bigquery
SELECT scenario_id, method,
       ROUND(var_95_usd) AS var_95_usd, ROUND(var_99_usd) AS var_99_usd,
       ROUND(es_99_usd) AS es_99_usd, ROUND(var_99_pct_notional, 2) AS var_99_pct
FROM `serving.portfolio_var_summary`
ORDER BY method, scenario_id

The distribution behind the baseline Monte Carlo numbers, with the 99% VaR and ES cut-offs marked:

In [ ]:
%%bigquery baseline_paths
SELECT path_id, SUM(sim_pnl_usd) AS pnl_usd
FROM `serving.mc_simulated_pnl`
WHERE scenario_id = 'baseline'
GROUP BY path_id

In [ ]:
pnl = baseline_paths["pnl_usd"]
var_99 = -pnl.quantile(0.01)
es_99 = -pnl[pnl <= -var_99].mean()

fig = px.histogram(baseline_paths, x="pnl_usd", nbins=200, color_discrete_sequence=[LIGHT],
                   title="Simulated 1-day portfolio P&L, 100,000 paths (baseline)")
fig.add_vrect(x0=pnl.min(), x1=-var_99, fillcolor=RED, opacity=0.08, line_width=0)
fig.add_vline(x=-var_99, line=dict(color=RED, width=2),
              annotation=dict(text=f"99% VaR ${var_99 / 1e6:.2f}M", font_color=RED), annotation_position="top right")
fig.add_vline(x=-es_99, line=dict(color="#202124", width=2, dash="dash"),
              annotation=dict(text=f"99% ES ${es_99 / 1e6:.2f}M"), annotation_position="top left")
fig.update_traces(hovertemplate="P&L around $%{x:,.0f}<br>%{y:,} paths<extra></extra>")
fig.update_layout(height=400, bargap=0.02, xaxis=dict(title="1-day P&L", tickformat="$~s"), yaxis_title="Paths")
fig.show()

Historical VaR is higher. The days that drive it:

In [ ]:
%%bigquery
SELECT date, ROUND(pnl_usd) AS pnl_usd, worst_day_rank
FROM `serving.historical_pnl`
ORDER BY worst_day_rank
LIMIT 5

**Reading the chart:** every bar is one real day from the last 500, replayed on today's portfolio. The red line is the 99% historical VaR: only 5 of the 500 days lose more. The three biggest losses are labelled.

In [ ]:
%%bigquery hist
SELECT date, pnl_usd, worst_day_rank
FROM `serving.historical_pnl`
ORDER BY date

In [ ]:
hist["date"] = pd.to_datetime(hist["date"])
var_hist = -hist["pnl_usd"].quantile(0.01)
worst = hist.nsmallest(3, "pnl_usd").sort_values("date")

fig = go.Figure(go.Bar(x=hist["date"], y=hist["pnl_usd"], marker_color=[RED if x < -var_hist else GREY for x in hist["pnl_usd"]], marker_line_width=0,
                       hovertemplate="%{x|%b %d, %Y}: $%{y:,.0f}<extra></extra>"))
fig.add_hline(y=-var_hist, line=dict(color=RED, dash="dash", width=1.5),
              annotation=dict(text=f"99% historical VaR ${var_hist / 1e6:.2f}M", font_color=RED), annotation_position="bottom right")
fig.add_annotation(x=worst["date"].iloc[0], y=worst["pnl_usd"].min(), ax=140, ay=0, xanchor="left", arrowhead=2, arrowcolor="#5F6368",
                   text="3 worst days: " + ", ".join(f"{d:%b} {d.day}" for d in worst["date"]) + f" {worst['date'].iloc[0]:%Y}")
fig.update_layout(title="The last 500 trading days, replayed on today's portfolio", height=400, bargap=0, showlegend=False,
                  yaxis=dict(title=None, tickformat="$~s"), xaxis_title=None)
fig.show()

Two things separate the numbers. The historical view uses the last 500 trading days and the simulation draws from the last 1,000, so the April 2025 tariff selloff weighs more in the historical tail. That window difference is most of the gap. The simulation also rescales each day to TimesFM's volatility, which accounts for a smaller part.

## Step 5: Add a stress scenario and simulate it

Ask a what-if: Information Technology falls 10% tomorrow. The next cell adds the scenario and runs the simulation procedure for it, writing 3,000,000 rows. If the scenario already exists, it is replaced.

In [ ]:
%%bigquery
DELETE FROM `raw.stress_scenarios` WHERE scenario_id = 'tech_down_10';

INSERT INTO `raw.stress_scenarios` (scenario_id, description, sector, shock_pct)
VALUES ('tech_down_10', 'Information Technology falls 10%', 'Information Technology', -0.10);

CALL `serving.simulate_scenario`('tech_down_10');

In [ ]:
%%bigquery by_desk
SELECT scenario_id, desk, ROUND(var_99_usd) AS var_99_usd
FROM `serving.var_by_desk`
WHERE scenario_id IN ('baseline', 'tech_down_10')
ORDER BY desk, scenario_id

In [ ]:
d = by_desk.pivot(index="desk", columns="scenario_id", values="var_99_usd")
change = d["tech_down_10"] - d["baseline"]
fig = go.Figure([
    go.Bar(name="Baseline", x=d.index, y=d["baseline"], marker_color=GREY,
           hovertemplate="%{x}<br>Baseline: $%{y:,.0f}<extra></extra>"),
    go.Bar(name="Tech falls 10%", x=d.index, y=d["tech_down_10"], marker_color=RED,
           text=[f"+${c / 1e6:.2f}M" if c > 1 else "no change" for c in change], textposition="outside", cliponaxis=False,
           hovertemplate="%{x}<br>Tech falls 10%: $%{y:,.0f}<extra></extra>"),
])
fig.update_layout(title="99% VaR by desk: only the desk holding Tech moves", barmode="group", height=420,
                  yaxis=dict(title=None, tickformat="$~s", range=[0, d.values.max() * 1.2]), xaxis_title=None)
fig.show()
d

Only Growth Equities moves, because it holds all the Information Technology stocks. GOOGL is Communication Services under GICS, so this shock does not hit it. The other desks are unchanged to the dollar: every scenario replays the same draws on the same paths.

The shock is applied on top of every path, so this is VaR given the shock: baseline risk plus the shock's loss. It is not Basel stressed VaR, which recalibrates to a stressed period.

## Step 6: Ask the data agent

The data agent answers in plain English and shows the SQL it ran. You can also chat with it in the console under **BigQuery > Agents > Pulsar Capital market risk**. This cell asks it through the Conversational Analytics API.

In [ ]:
from google.auth.transport.requests import AuthorizedSession
from google.cloud import bigquery
from IPython.display import Markdown, display

QUESTION = "How does the tech_down_10 scenario change our 99% VaR by desk?"

credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
session = AuthorizedSession(credentials)
API = f"https://geminidataanalytics.googleapis.com/v1/projects/{PROJECT_ID}/locations/global"

# OpenTofu gives the agent a new ID whenever its definition changes, so find it
# by its display name.
agents = session.get(f"{API}/dataAgents").json().get("dataAgents", [])
agent = next(a["name"] for a in agents if a.get("displayName") == "Pulsar Capital market risk")

response = session.post(
    f"{API}:chat",
    json={
        "messages": [{"userMessage": {"text": QUESTION}}],
        "dataAgentContext": {"dataAgent": agent},
    },
    timeout=300,
)
response.raise_for_status()

# The agent returns its SQL as generatedSql when it writes a new query. When it
# starts from a verified query, it returns only the BigQuery job it ran, so the
# SQL is read from that job.
sql, answer = [], []
for message in response.json():
    system = message.get("systemMessage", {})
    data = system.get("data", {})
    if data.get("generatedSql"):
        sql.append(data["generatedSql"])
    if data.get("bigQueryJob"):
        job = data["bigQueryJob"]
        sql.append(bigquery.Client(project=PROJECT_ID).get_job(job["jobId"], location=job.get("location")).query)
    if system.get("text", {}).get("textType") == "FINAL_RESPONSE":
        answer.extend(system["text"].get("parts", []))

print("SQL the agent ran:")
print(sql[-1] if sql else "(none returned)")
display(Markdown("**Answer:** " + " ".join(answer)))

### Check the answer against the view

Change `QUESTION` above to *Which 5 stocks contribute the most tail risk?* and rerun it. The chart below reads the same view the agent uses, so you can check its answer. Each stock's share of the tail (99% ES) sits next to its share of the portfolio. A red bar much longer than its grey bar is concentration.

In [ ]:
%%bigquery contrib
SELECT symbol, desk, notional_usd, es_99_contribution_usd, share_of_es_99_pct
FROM `serving.var_contributions`
WHERE scenario_id = 'baseline'
ORDER BY es_99_contribution_usd DESC
LIMIT 10

In [ ]:
c = contrib.assign(notional_pct=100 * contrib["notional_usd"] / 100_000_000)
top = c.iloc[0]
fig = go.Figure([
    go.Bar(name="Share of 99% ES (tail loss)", y=c["symbol"], x=c["share_of_es_99_pct"], orientation="h", marker_color=RED,
           hovertemplate="%{y}: %{x:.1f}% of the tail loss<extra></extra>"),
    go.Bar(name="Share of notional", y=c["symbol"], x=c["notional_pct"], orientation="h", marker_color=LIGHT,
           hovertemplate="%{y}: %{x:.1f}% of the money<extra></extra>"),
])
fig.update_layout(title=f"{top['symbol']}: {top['notional_pct']:.0f}% of the money, {top['share_of_es_99_pct']:.1f}% of the tail risk",
                  barmode="group", height=480, xaxis=dict(title=None, ticksuffix="%"), yaxis=dict(title=None, autorange="reversed"))
fig.show()

## Reset

Remove the scenario you added, so the next run starts from the seeded `baseline` and `energy_down_15`.

In [ ]:
%%bigquery
DELETE FROM `serving.mc_simulated_pnl` WHERE scenario_id = 'tech_down_10';
DELETE FROM `raw.stress_scenarios` WHERE scenario_id = 'tech_down_10';